In [2]:
import pandas as pd
import os
import subprocess
import pandas as pd
import shutil
from difflib import SequenceMatcher
import ast
import math
from collections import Counter

In [3]:
REPO_BASE = "https://github.com"
LOCAL_DIR = "repos"

# 🔥 MUST initialize these
current_repo_path = None
current_repo_key = None

In [ ]:
def run(cmd, cwd=None):
    result = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    return result.stdout.strip()

def ensure_repo(source, repo):
    global current_repo_path, current_repo_key

    repo_key = f"{source}_{repo}"
    path = os.path.join(LOCAL_DIR, repo_key)

    # 🔥 delete previous repo if switching
    if current_repo_path and current_repo_key != repo_key:
        print(f"Deleting previous repo: {current_repo_key}")
        shutil.rmtree(current_repo_path, ignore_errors=True)

    # update tracker
    current_repo_path = path
    current_repo_key = repo_key

    if not os.path.exists(LOCAL_DIR):
        os.makedirs(LOCAL_DIR)

    if not os.path.exists(path):
        url = f"{REPO_BASE}/{source}/{repo}.git"
        print(f"Cloning {url}")
        subprocess.run(["git", "clone", "--quiet", url, path])

    return path


In [ ]:
def get_changed_files(repo_path, commit_hash):
    out = run(
        ["git", "diff-tree", "--no-commit-id", "--name-only", "-r", commit_hash],
        cwd=repo_path
    )
    return out.split("\n") if out else []

def get_file(repo_path, commit_hash, file_path):
    try:
        return run(["git", "show", f"{commit_hash}:{file_path}"], cwd=repo_path)
    except:
        return ""

In [ ]:
def compute_similarity(before, after):
    return SequenceMatcher(None, before, after).ratio()

In [ ]:
def get_ast_entropy(code: str):
    if not code or not code.strip():
        return 0

    try:
        tree = ast.parse(code)
    except:
        return 0  # invalid / non-python code

    node_types = []

    for node in ast.walk(tree):
        node_types.append(type(node).__name__)

    if not node_types:
        return 0

    freq = Counter(node_types)
    total = len(node_types)

    entropy = 0
    for c in freq.values():
        p = c / total
        entropy -= p * math.log2(p)

    return entropy

In [ ]:
def process_dataframe(df):
    results = []

    for i, row in df.iterrows():
        source = row["source"]
        repo = row["repo"]
        commit_hash = row["hash"]
        parents = row["parents"]

        if pd.isna(parents) or parents == "":
            continue

        parent_list = [p.strip() for p in str(parents).split(";")]

        repo_path = ensure_repo(source, repo)

        files = get_changed_files(repo_path, commit_hash)

        for parent_hash in parent_list:
            for f in files:

                # OPTIONAL: filter only code files
                if not f.endswith((".java", ".py", ".js")):
                    continue

                before = get_file(repo_path, parent_hash, f)
                after = get_file(repo_path, commit_hash, f)

                results.append({
                    "source": source,
                    "repo": repo,
                    "hash": commit_hash,
                    "parent": parent_hash,
                    "file": f,
                    "preceding_code": before,
                    "succeeding_code": after
                })

        if i % 10 == 0:
            print(f"Processed {i} rows")

        if i >= 100:  # 🔥 limit for testing    
            break

    return pd.DataFrame(results)


# Example usage
if __name__ == "__main__":
    ##df = pd.read_csv("input.csv")   # or already loaded dataframe

    output_df = process_dataframe(df_test)
    print(output_df.head())

    output_df.to_csv("output.csv", index=False)